# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias, ClassVar

from tsfresh import extract_features, select_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, MinimalFCParameters, IndexBasedFCParameters
from tsfresh.utilities.dataframe_functions import impute

BASE_PATH = "../../.."
DATASET_SEPARATOR = ";"
DATASET = f"{BASE_PATH}/experiment-data"
DATASET_FILENAME = f"{DATASET}/flat_dataset.alltasks.csv.gz"
LABELS_SEPARATOR = ","
LABELS = f"{DATASET}/labels.csv"

EXTRACTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_extfeat.csv"
SELECTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_selfeat.csv"

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": True,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": True,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}

@dataclass
class FullData:
    subject: ClassVar[list[int]] = []
    task: ClassVar[list[str]] = []
    timestamp: ClassVar[list[float]] = []
    eda: ClassVar[list[float]] = []
    ppg: ClassVar[list[float]] = []
    accel_x: ClassVar[list[float]] = []
    accel_y: ClassVar[list[float]] = []
    accel_z: ClassVar[list[float]] = []
    gyro_x: ClassVar[list[float]] = []
    gyro_y: ClassVar[list[float]] = []
    gyro_z: ClassVar[list[float]] = []
    temp: ClassVar[list[float]] = []
    pressure: ClassVar[list[float]] = []
    b_classes: ClassVar[list[int]] = []
    t_classes: ClassVar[list[int]] = []
    q_classes: ClassVar[list[int]] = []

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            {
                "Subject": self.subject,
                "Task": self.task,
                "Timestamp": self.timestamp,
                "EDA": self.eda,
                "PPG": self.ppg,
                "Accel_X": self.accel_x,
                "Accel_Y": self.accel_y,
                "Accel_Z": self.accel_z,
                "Gyro_X": self.gyro_x,
                "Gyro_Y": self.gyro_y,
                "Gyro_Z": self.gyro_z,
                "Temperature": self.temp,
                "Pressure": self.pressure,
                "Bin_Class": self.b_classes,
                "Ter_Class": self.t_classes,
                "Qad_Class": self.q_classes,
            }
        )

    def assert_lengths(self) -> None:
        differs = False
        lengths = {}
        expected_len = len(self.timestamp)
        for name, instance_attr in self.__dict__.items():
            lengths[name] = len(instance_attr)
            if len(instance_attr) != expected_len:
                differs = True
        if differs:
            AssertionError(f"Columns are not the same lenght: {lengths}")

### Build dataset from data files

In [28]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
tasks: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        tasks[key] = labels_df[conf["col_name"]]

display(tasks["b"])
tasks_with_labels: list[str] = tasks["b"].index


subject/task
01-AmusementClip    0
01-Baseline         0
01-EmoReset         0
01-FormL            1
01-FormM            1
                   ..
21-Baseline         0
21-EmoReset         0
21-FormL            0
21-FormM            0
21-StressClip       1
Name: binary-stress, Length: 126, dtype: int64

In [29]:
# Creating dataset file, it may be skipped

# data will not be normalized or standarized
# using calibrated data (no raw data since Shimmer3's GSR sensor requires range scaling)
CREATE_DATASET_FILE = False
EXPECTED_NUM_FILES = 21
SAMPLING_RATE = 51.2
NEUROCLEAN = True

if CREATE_DATASET_FILE:
    import glob
    import neurokit2 as nk
    type FeatureDict = dict[str, np.ndarray]

    col_types = {
        "Timestamp": float,
        "Event": str,
        "ExtraEvent": str,
        "AccelLN_X": float,
        "AccelLN_Y": float,
        "AccelLN_Z": float,
        "Battery": float,
        "GSR_Range": int,
        "Skin_Conductance": float,
        "Skin_Resistance": float,
        "Gyro_X": float,
        "Gyro_Y": float,
        "Gyro_Z": float,
        "PPG": float,
        "Pressure": float,
        "Temperature": float,
        "AccelLN_X_Uncal": int,
        "AccelLN_Y_Uncal": int,
        "AccelLN_Z_Uncal": int,
        "Skin_Conductance_Uncal": int,
        "PPG_Uncal": int,
    }

    ####### LOAD DATA
    filelist = glob.glob(f"{DATASET}/*.Annotated.csv")
    filelist.sort()
    if len(filelist) != EXPECTED_NUM_FILES:
        raise ValueError(f"Expected {EXPECTED_NUM_FILES} files, found: {len(filelist)}")

    full_data = FullData()
    subject_to_int: dict[str, int] = {}
    subject_counter = 0

    def split_by_task(df: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
        output: list[tuple[str, pd.DataFrame]] = []
        tasks = ["Baseline", "AmusementClip", "StressClip", "EmoReset", "FormL", "FormM", "Debriefing"]
        start_idx = end_idx = 0
        for task in tasks:
            if task == "FormL":
                if "FormLRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormLRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "L15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[-1])
            elif task == "FormM":
                if "FormMRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormMRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "M15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[-1])
            else:
                start_idx = df.index.get_loc(df[df["Event"] == task].index[0])
                end_idx = df.index.get_loc(df[df["Event"] == task].index[-1])
            output.append((task, df[start_idx:end_idx]))
        return output


    for item in filelist:
        file: pd.DataFrame = pd.read_csv(
            item,
            delimiter=";",
            date_format=r"%Y-%m-%d %H:%M:%S.%f",
            parse_dates=["Datetime", "Timestamp"],
            index_col=["Datetime"],
            dtype=col_types,
        )
        filename = item.split("/")[-1]
        subject_id = filename.split("-")[1]
        if subject_id not in subject_to_int:
            subject_to_int[subject_id] = subject_counter
            subject_counter += 1

        for task, event_df in split_by_task(file):
            label_id = f"{subject_id}-{task}"
            if label_id not in tasks_with_labels:
                continue
            n_elements = event_df["Timestamp"].size
            ppg_signal = (
                nk.ppg_clean(np.array(event_df["PPG"]), sampling_rate=SAMPLING_RATE)
                if NEUROCLEAN
                else event_df["PPG"].to_list()
            )
            eda_signal = (
                nk.eda_clean(np.array(event_df["Skin_Conductance"]), sampling_rate=SAMPLING_RATE, method="neurokit")
                if NEUROCLEAN
                else event_df["Skin_Conductance"].to_list()
            )
            if ppg_signal.size != n_elements:
                raise Exception(f"Sizes differ {ppg_signal.size} vs {n_elements}")
            full_data.subject.extend(np.full(n_elements, subject_to_int[subject_id]))
            full_data.task.extend(np.full(n_elements, task))
            full_data.timestamp.extend(event_df["Timestamp"].to_list())
            full_data.ppg.extend(ppg_signal)
            full_data.eda.extend(eda_signal)
            full_data.accel_x.extend(event_df["AccelLN_X"].to_list())
            full_data.accel_y.extend(event_df["AccelLN_Y"].to_list())
            full_data.accel_z.extend(event_df["AccelLN_Z"].to_list())
            full_data.gyro_x.extend(event_df["Gyro_X"].to_list())
            full_data.gyro_y.extend(event_df["Gyro_Y"].to_list())
            full_data.gyro_z.extend(event_df["Gyro_Z"].to_list())
            full_data.temp.extend(event_df["Temperature"].to_list())
            full_data.pressure.extend(event_df["Pressure"].to_list())
            full_data.b_classes.extend(np.full(n_elements, tasks["b"][label_id]))
            full_data.t_classes.extend(np.full(n_elements, tasks["t"][label_id]))
            full_data.q_classes.extend(np.full(n_elements, tasks["q"][label_id]))
            full_data.assert_lengths()

    fulldata_df = full_data.to_dataframe()
    fulldata_df.to_csv(DATASET_FILENAME, sep=DATASET_SEPARATOR, index=False, compression="gzip")
    display(fulldata_df)


In [30]:
DESIRED_COLUMNS = ["Task", "Timestamp", "EDA", "PPG"]
DESIRED_LABELS = "Bin_Class"

In [31]:
if not CREATE_DATASET_FILE:
    fulldata_df = pd.read_csv(DATASET_FILENAME, sep=DATASET_SEPARATOR, compression="gzip")
    X = fulldata_df[DESIRED_COLUMNS]
    y = fulldata_df[DESIRED_LABELS]
    X_groups = fulldata_df["Subject"]

In [32]:
display(X)
display(y)
display(X_groups)

,Task,Timestamp,EDA,PPG
0,Baseline,1.749609e+12,1.397531,-8.688205
1,Baseline,1.749609e+12,1.403573,-17.615588
2,Baseline,1.749609e+12,1.409202,-26.264472
3,Baseline,1.749609e+12,1.414017,-34.198909
4,Baseline,1.749609e+12,1.417698,-40.997525
...,...,...,...,...
5740079,FormM,1.752561e+12,6.794845,41.209776
5740080,FormM,1.752561e+12,6.801706,5.093412
5740081,FormM,1.752561e+12,6.808988,-24.224583
5740082,FormM,1.752561e+12,6.816556,-42.873762


0          0
1          0
2          0
3          0
4          0
          ..
5740079    0
5740080    0
5740081    0
5740082    0
5740083    0
Name: Bin_Class, Length: 5740084, dtype: int64

0           0
1           0
2           0
3           0
4           0
           ..
5740079    20
5740080    20
5740081    20
5740082    20
5740083    20
Name: Subject, Length: 5740084, dtype: int64

In [ ]:
features = extract_features(
    X,
    column_id="Task",
    column_sort="Timestamp",
    impute_function=impute,
    default_fc_parameters=EfficientFCParameters()
)

Feature Extraction:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
# features.to_csv(EXTRACTED_FEATURES_FILENAME, index=False)
# features = pd.read_csv(sel_features_filename, index_col=0)
display(features)


,ECG__variance_larger_than_standard_deviation,ECG__has_duplicate_max,ECG__has_duplicate_min,ECG__has_duplicate,ECG__sum_values,ECG__abs_energy,ECG__mean_abs_change,ECG__mean_change,ECG__mean_second_derivative_central,ECG__median,ECG__mean,ECG__length,ECG__standard_deviation,ECG__variation_coefficient,ECG__variance,ECG__skewness,ECG__kurtosis,ECG__root_mean_square,ECG__absolute_sum_of_changes,ECG__longest_strike_below_mean,ECG__longest_strike_above_mean,ECG__count_above_mean,ECG__count_below_mean,ECG__last_location_of_maximum,ECG__first_location_of_maximum,ECG__last_location_of_minimum,ECG__first_location_of_minimum,ECG__percentage_of_reoccurring_values_to_all_values,ECG__percentage_of_reoccurring_datapoints_to_all_datapoints,ECG__sum_of_reoccurring_values,ECG__sum_of_reoccurring_data_points,ECG__ratio_value_number_to_time_series_length,ECG__maximum,ECG__absolute_maximum,ECG__minimum,ECG__benford_correlation,ECG__time_reversal_asymmetry_statistic__lag_1,ECG__time_reversal_asymmetry_statistic__lag_2,ECG__time_reversal_asymmetry_statistic__lag_3,ECG__c3__lag_1,...,EDA__number_crossing_m__m_1,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_0,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_1,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_2,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_3,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_4,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_5,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_6,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_7,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_8,EDA__energy_ratio_by_chunks__num_segments_10__segment_focus_9,EDA__ratio_beyond_r_sigma__r_0.5,EDA__ratio_beyond_r_sigma__r_1,EDA__ratio_beyond_r_sigma__r_1.5,EDA__ratio_beyond_r_sigma__r_2,EDA__ratio_beyond_r_sigma__r_2.5,EDA__ratio_beyond_r_sigma__r_3,EDA__ratio_beyond_r_sigma__r_5,EDA__ratio_beyond_r_sigma__r_6,EDA__ratio_beyond_r_sigma__r_7,EDA__ratio_beyond_r_sigma__r_10,EDA__count_above__t_0,EDA__count_below__t_0,EDA__lempel_ziv_complexity__bins_2,EDA__lempel_ziv_complexity__bins_3,EDA__lempel_ziv_complexity__bins_5,EDA__lempel_ziv_complexity__bins_10,EDA__lempel_ziv_complexity__bins_100,EDA__fourier_entropy__bins_2,EDA__fourier_entropy__bins_3,EDA__fourier_entropy__bins_5,EDA__fourier_entropy__bins_10,EDA__fourier_entropy__bins_100,EDA__permutation_entropy__dimension_3__tau_1,EDA__permutation_entropy__dimension_4__tau_1,EDA__permutation_entropy__dimension_5__tau_1,EDA__permutation_entropy__dimension_6__tau_1,EDA__permutation_entropy__dimension_7__tau_1,EDA__query_similarity_count__query_None__threshold_0.0,EDA__mean_n_absolute_max__number_of_maxima_7
2ea4_Breathing,1.0,0.0,0.0,0.0,-3.856781e+04,4.485348e+10,74.609834,-0.332483,0.009590,-8.835560,-1.285594e+00,30000.0,1222.748650,-9.511160e+02,1.495114e+06,4.120171,26.832356,1222.749325,2.238220e+06,199.0,240.0,14805.0,15195.0,0.665500,0.665467,0.613900,0.613867,0.0,0.0,0.0,0.0,1.0,11901.974647,11901.974647,-2217.463250,0.989362,-4.970670e+07,-9.858502e+07,-1.639310e+08,7.095612e+09,...,0.0,0.100694,0.101929,0.098300,0.098257,0.097502,0.100328,0.101486,0.097795,0.099892,0.103818,0.627167,0.188533,0.108933,0.056133,0.024467,0.018967,0.0,0.0,0.0,0.0,1.0,0.0,0.010600,0.012367,0.016133,0.022767,0.078200,0.045395,0.045395,0.045395,0.045395,0.136002,0.681340,0.690385,0.699430,0.708473,0.717516,0.0,33840.828474
2ea4_Counting1,1.0,0.0,0.0,0.0,9.022187e-10,4.920041e+10,74.552268,0.001808,0.000567,13.187743,3.007396e-14,30000.0,1280.630119,4.258269e+16,1.640014e+06,3.817196,20.069715,1280.630119,2.236493e+06,199.0,203.0,15333.0,14667.0,0.857333,0.857300,0.903967,0.903933,0.0,0.0,0.0,0.0,1.0,10398.125644,10398.125644,-1869.901199,0.991803,-4.007354e+06,-3.098913e+07,-9.880246e+07,7.622836e+09,...,0.0,0.070608,0.115859,0.115910,0.115910,0.114698,0.094532,0.076830,0.079422,0.107459,0.108772,0.790700,0.207633,0.097233,0.058400,0.006733,0.000000,0.0,0.0,

In [27]:
sel_features_filename = "../Features/tfresh_eda_cga_features.efficient.selected.csv"
# X_selected.to_csv(sel_features_filename)
X_selected = pd.read_csv(sel_features_filename, index_col=0)
display(X_selected)

,"ECG__fft_coefficient__attr_""abs""__coeff_0","ECG__fft_aggregated__aggtype_""skew""","EDA__augmented_dickey_fuller__attr_""usedlag""__autolag_""AIC""","ECG__fft_aggregated__aggtype_""kurtosis""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""min""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""mean""","EDA__linear_trend__attr_""stderr""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""mean""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_10__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_5__f_agg_""max""","EDA__agg_linear_trend__attr_""stderr""__chunk_len_50__f_agg_""max""","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.6__ql_0.4","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.2","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.2","ECG__change_quantiles__f_agg_""var""__isabs_True__qh_0.6__ql_0.4","EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.0","EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.2__ql_0.0",EDA__absolute_sum_of_changes,"EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_1.0__ql_0.0",EDA__mean_abs_change,"EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.4__ql_0.0","EDA__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.0","EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.6__ql_0.0","EDA__change_quantiles__f_agg_""var""__isabs_True__qh_0.4__ql_0.0","EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.4","EDA__change_quantiles__f_agg_""var""__isabs_False__qh_0.4__ql_0.0","ECG__fft_aggregated__aggtype_""variance""","EDA__change_quantiles__f_agg_""var""__isabs_True__qh_0.8__ql_0.0","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.4","EDA__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.6","EDA__change_quantiles__f_agg_""var""__isabs_False__qh_0.6__ql_0.0","ECG__change_quantiles__f_agg_""mean""__isabs_True__qh_0.8__ql_0.4","EDA__change_quantiles__f_agg_""var""__isabs_False__qh_0.2__ql_0.0","ECG__fft_coefficient__attr_""abs""__coeff_38",...,"EDA__fft_coefficient__attr_""angle""__coeff_16","EDA__fft_coefficient__attr_""angle""__coeff_11",EDA__binned_entropy__max_bins_10,"EDA__fft_coefficient__attr_""real""__coeff_95","EDA__fft_coefficient__attr_""angle""__coeff_49","ECG__change_quantiles__f_agg_""var""__isabs_False__qh_1.0__ql_0.8",EDA__ar_coefficient__coeff_10__k_10,"EDA__fft_coefficient__attr_""angle""__coeff_34","EDA__fft_coefficient__attr_""real""__coeff_71","ECG__cwt_coefficients__coeff_4__w_5__widths_(2, 5, 10, 20)",ECG__sum_values,ECG__mean,"ECG__change_quantiles__f_agg_""var""__isabs_False__qh_1.0__ql_0.0",ECG__cid_ce__normalize_False,EDA__ar_coefficient__coeff_5__k_10,"ECG__fft_coefficient__attr_""real""__coeff_0",EDA__ar_coefficient__coeff_7__k_10,"ECG__fft_coefficient__attr_""abs""__coeff_94","ECG__agg_linear_trend__attr_""intercept""__chunk_len_10__f_agg_""var""",EDA__lempel_ziv_complexity__bins_10,"ECG__agg_linear_trend__attr_""intercept""__chunk_len_5__f_agg_""var""","ECG__fft_coefficient__attr_""abs""__coeff_55","ECG__fft_coefficient__attr_""real""__coeff_21","ECG__fft_coefficient__attr_""abs""__coeff_80","EDA__fft_coefficient__attr_""angle""__coeff_37","ECG__cwt_coefficients__coeff_3__w_5__widths_(2, 5, 10, 20)",EDA__lempel_ziv_complexity__bins_100,"ECG__change_quantiles__f_agg_""var""__isabs_True__qh_1.0__ql_0.8","EDA__fft_coefficient__attr_""abs""__coeff_44","EDA__fft_coefficient__attr_""abs""__coeff_51","EDA__fft_coefficient__attr_""angle""__coeff_15","ECG__change_quantiles__f_agg_""mean""__is

In [29]:
# RFECV

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

RAN_STATE = 21
SPLITS = 10

estimator = RandomForestClassifier(max_depth=3, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(X_selected, y)
features_mask = selector.support_
X_selected_rfe = X_selected.loc[:, features_mask]

features_scores = { "scores": [], "features": [] }
for score, feat in zip(selector.estimator_.feature_importances_, X_selected_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected tsfresh features scores for StressID")
display(features_scores_df)

KeyboardInterrupt: 

In [31]:
# Divergence analysis on StressID
import plotly.express as px

perplexity = np.arange(15, 210, 15)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(X_selected)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [32]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10)
si_Ncomp = 2
si_Perp = 190

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter(x=si_X_tsne[:,0], y=si_X_tsne[:,1], color=y, width=800, height=600)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

0.05135606229305267

In [33]:
# t-SNE in StressID
# Best N-comp=3, Perp=90 np.arange(15, 210, 15)
si_Ncomp = 3
si_Perp = 90

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter_3d(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:,2], color=y, opacity=0.7, width=800, height=600)
fig.update_layout(title="t-SNE visualization of StressID dataset")
fig.show()

0.05272906273603439